# Sovereign AI: Fine-Tuning TinyAya on Yoruba for Constrained Kubernetes

This notebook trains a localized LoRA adapter on top of **CohereLabs/tiny-aya-earth** (3.35B parameters) using the **masakhane/african-ultrachat** (Yoruba split) dataset.

> **Tip:** Run on Google Colab with GPU enabled: **Runtime -> Change runtime type -> T4 GPU or A100**.

In [1]:
# 1. Install rock-solid core dependencies (without fragile external wrappers)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q -U "transformers>=4.44.0" "peft>=0.12.0" "datasets>=2.20.0" "accelerate>=0.33.0" bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 80.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 40.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 82.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 84.5 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confli

In [2]:
# 2. Verify GPU allocation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> select T4 or A100 GPU.")

PyTorch version: 2.10.0+cu128
CUDA Available: True
GPU Device: Tesla T4
VRAM: 15.64 GB


In [3]:
# 3. Secure Hugging Face Authentication (Supports Kaggle Secrets, Colab Secrets, & Manual Input)
import os
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or input("Enter your Hugging Face Access Token: ").strip()

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("Authenticated to Hugging Face successfully!")

Authenticated to Hugging Face successfully via Kaggle Secrets!


In [4]:
# 4. Load Base Model and Run Baseline Evaluation (Before Training)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "CohereLabs/tiny-aya-earth"
compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

# 4-bit Quantization Config for Colab GPU
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
else:
    bnb_config = None

print(f"Loading base tokenizer and model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    torch_dtype=compute_dtype,
    token=True,
    trust_remote_code=True,
)

# Baseline Generation Test
test_prompt = "<|START_OF_TURN_TOKEN|><|USER_TOKEN|>Bawo ni o se le se alaye bi ero ayelujara (Internet) se n sise ni ede Yoruba to rorun?<|END_OF_TURN_TOKEN|><|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>"
device = "cuda" if torch.cuda.is_available() else "cpu"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

print("Generating baseline output...")
with torch.no_grad():
    outputs = base_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

baseline_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- BASELINE OUTPUT (Before Fine-Tuning) ---")
print(baseline_response)
print("-------------------------------------------\n")

Loading base tokenizer and model: CohereLabs/tiny-aya-earth...


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Generating baseline output...

--- BASELINE OUTPUT (Before Fine-Tuning) ---
**Bawo ni o se le se alaye bi ero ayelujara (Internet) se n sise ni ede Yoruba to rorun?**

Internet ni eré t’ọjọ́wọ́, tó jẹ́ àkọ́kọ́ láti fún àwọn ènìyàn láti wọ lé àwọn ìtòjọ (pages), ìwé (documents), àti ìtọ́ka (links) lórí ẹ̀rọ ayélujára (internet). Àwọn ọ̀pọ̀lọpọ̀ nínú rẹ̀ jẹ́:

1. **Àwọn Ìtòjọ (Web Pages)**: Àwọn ojúewé tó ní àkóónú láti ṣàf
-------------------------------------------



In [5]:
# 5. Prepare and Tokenize African-UltraChat Yoruba Dataset
from datasets import load_dataset

dataset_name = "masakhane/african-ultrachat"
print(f"Loading dataset: {dataset_name} (Yoruba split)...")

try:
    dataset = load_dataset(dataset_name, "yo", split="train")
except Exception as e:
    print(f"Direct split load fallback: {e}")
    dataset = load_dataset(dataset_name, split="train")
    if "language" in dataset.column_names:
        dataset = dataset.filter(lambda x: x["language"].lower() in ["yo", "yoruba"])

# Subsample 3000 dialogues for efficient training
max_samples = min(600, len(dataset))
dataset = dataset.shuffle(seed=42).select(range(max_samples))
print(f"Training set ready with {len(dataset)} conversation records.")

def format_and_tokenize(batch):
    texts = []
    for record_messages in batch["messages"]:
        turn_text = ""
        for msg in record_messages:
            role = msg.get("role", "")
            content = msg.get("content", "").strip()
            if role == "user":
                turn_text += f"<|START_OF_TURN_TOKEN|><|USER_TOKEN|>{content}<|END_OF_TURN_TOKEN|>"
            elif role == "assistant":
                turn_text += f"<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>{content}<|END_OF_TURN_TOKEN|>"
        texts.append(turn_text)
    # max_length=512 prevents massive 262k vocabulary logits allocation on T4 GPU
    return tokenizer(texts, truncation=True, max_length=512, padding="max_length")

print("Tokenizing dataset (max_length=512)...")
tokenized_dataset = dataset.map(format_and_tokenize, batched=True, remove_columns=dataset.column_names)
print("Dataset tokenization complete!")


Loading dataset: masakhane/african-ultrachat (Yoruba split)...


README.md:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

Direct split load fallback: BuilderConfig 'yo' not found. Available: ['default']


train/Amharic.jsonl: reconstructing file:   0%|          |  0.00B / 52.9MB            

train/Amharic.jsonl: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/1100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53900 [00:00<?, ? examples/s]

Training set ready with 600 conversation records.
Tokenizing dataset (max_length=512)...


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Dataset tokenization complete!


In [6]:
# 6. Automated LoRA Training (Finishes in ~7-8 minutes with auto-save)
import gc
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Clean up GPU VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

# Optimized for T4: max_steps=60, auto-saving checkpoints every 20 steps
training_args = TrainingArguments(
    output_dir="./tiny-aya-earth-yoruba-lora",
    max_steps=60,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    max_grad_norm=0.3,
    warmup_steps=15,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    args=training_args,
)

print("Starting automated 60-step LoRA training...")
trainer.train()

# Save final trained adapter
print("Saving final trained adapter to ./tiny-aya-earth-yoruba-lora...")
peft_model.save_pretrained("./tiny-aya-earth-yoruba-lora")
tokenizer.save_pretrained("./tiny-aya-earth-yoruba-lora")
print("SUCCESS: Adapter saved to ./tiny-aya-earth-yoruba-lora!")


trainable params: 30,228,480 || all params: 3,379,456,000 || trainable%: 0.8945
Starting automated 60-step LoRA training...


Step,Training Loss
10,2.295816
20,1.992184
30,1.774902
40,1.720317
50,1.651255
60,1.680459


Saving final trained adapter to ./tiny-aya-earth-yoruba-lora...
SUCCESS: Adapter saved to ./tiny-aya-earth-yoruba-lora!


In [7]:
# 7. Post-Training Evaluation: Test Identical Prompt
peft_model.eval()
print("Generating fine-tuned output with identical prompt...")
with torch.no_grad():
    outputs = peft_model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

finetuned_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- FINE-TUNED OUTPUT (After LoRA Training) ---")
print(finetuned_response)
print("-----------------------------------------------\n")

Generating fine-tuned output with identical prompt...

--- FINE-TUNED OUTPUT (After LoRA Training) ---
Mo ti gbọ pe o fẹ ki n ṣalaye bi ero ayelujara (Internet) ṣe n ṣiṣẹ ni èdè Yorùbá. Internet jẹ agbaye ayelujara to ṣe pataki fun gbogbo eniyan, nitori pe o jẹ irinṣẹ ti o rọrun lati lo fun gbogbo eniyan. O jẹ eto ti o ni awọn olupin ti o sopọ mọ lilo awọn aaye ori ayelujara, eyiti o jẹ awọn orisun alaye lori ayelujara. O ni awọn ojú opó òkèèrè bíi Google, YouTube, ati Facebook ti o jẹ ki eniyan le wadii awọn ohun elo, wo fidio, ati pin alaye pẹlu awọn elomiran. O tun ni awọn eto ibaraẹ
-----------------------------------------------



In [13]:
# 8. Create repo and upload adapter
from huggingface_hub import HfApi

api = HfApi()
repo_id = "HusseinAlamutu/tiny-aya-earth-yoruba-lora"

print(f"Creating repository on Hugging Face: {repo_id}...")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

print(f"Uploading trained adapter to {repo_id}...")
api.upload_folder(
    folder_path="./tiny-aya-earth-yoruba-lora",
    repo_id=repo_id,
    repo_type="model"
)
print(f"SUCCESS: Adapter is live at https://huggingface.co/{repo_id}!")

Creating repository on Hugging Face: HusseinAlamutu/tiny-aya-earth-yoruba-lora...
Uploading trained adapter to HusseinAlamutu/tiny-aya-earth-yoruba-lora...
SUCCESS: Adapter is live at https://huggingface.co/HusseinAlamutu/tiny-aya-earth-yoruba-lora!


In [15]:
# 9. Merge LoRA Weights into Base Model and Save for GGUF Conversion
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "CohereLabs/tiny-aya-earth"
merged_output_dir = "./tiny-aya-earth-yoruba-merged"

print("Consolidating weights: loading base model in FP16 on CPU...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu",
    token=True,
    trust_remote_code=True,
)

print("Loading trained LoRA adapter from disk...")
merged_model = PeftModel.from_pretrained(base_fp16, "./tiny-aya-earth-yoruba-lora")
merged_model = merged_model.merge_and_unload()

print(f"Saving merged weights to {merged_output_dir}...")
merged_model.save_pretrained(merged_output_dir)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=True, trust_remote_code=True)
tokenizer.save_pretrained(merged_output_dir)
print("SUCCESS: Fully merged weights saved to ./tiny-aya-earth-yoruba-merged!")

Consolidating weights: loading base model in FP16 on CPU...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading trained LoRA adapter from disk...
Saving merged weights to ./tiny-aya-earth-yoruba-merged...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SUCCESS: Fully merged weights saved to ./tiny-aya-earth-yoruba-merged!
